# Módulo 6: Classificação com Regressão Logística e Indicadores de Desempenho

> **Resumo Didático Integrado** — Baseado no *Capítulo 3 (Análise de Dados — Seção 3: Regressão Logística & Biblioteca de Avaliação)*.
> 
> Este caderno é **100% autocontido**: você aprenderá a diferença teórica entre regressão e classificação, a formulação da **função Sigmóide/Logística**, o ajuste via `glm(family = binomial)`, a geração de probabilidades, o impacto do limiar de decisão (*threshold*), o cálculo completo da **Matriz de Confusão** e das métricas clínicas (**Sensitividade, Especificidade, Precisão, Log-Loss**), finalizando com a construção e análise da **Curva ROC e AUC**.

---

## 1. Visão Geral: Problemas de Classificação vs. Regressão

Na **Regressão Linear**, a variável resposta $Y$ é contínua e pode assumir qualquer valor real (ex.: preço de imóveis, temperatura).
Na **Classificação Binária**, o desfecho de interesse é categórico e dicotômico:

$$Y \in \{0, 1\}$$

Exemplos: *A água é potável ($1$) ou contaminada ($0$)?*, *O cliente pagará a fatura ($1$) ou ficará inadimplente ($0$)?*.

### Por que a Regressão Linear Falha na Classificação?
1. **Extrapolação de Probabilidades**: A reta pode prever valores menores que $0$ ou maiores que $1$, o que viola os axiomas de probabilidade.
2. **Heterocedasticidade Severa**: A variância de uma variável de Bernoulli depende da probabilidade ($Var(Y) = p(1-p)$), violando o pressuposto de variância constante dos resíduos.
3. **Sensibilidade a Outliers**: Observações extremas puxam a reta e alteram o ponto de corte de forma inconsistente.

---

## 2. A Função Logística e o Conceito de *Log-Odds*

Para garantir que as saídas do modelo representem probabilidades legítimas restritas ao intervalo $(0, 1)$, utilizamos a **Função Sigmóide / Logística**:

$$p(X) = P(Y = 1 \mid X) = \frac{1}{1 + e^{-z}} = \frac{e^z}{1 + e^z}$$

Onde $z = \beta_0 + \beta_1 X_1 + \dots + \beta_k X_k$.

Ao isolar a razão entre a probabilidade de ocorrência e de não ocorrência (as **chances** ou *odds*):

$$\text{Odds} = \frac{p}{1 - p}$$

Aplicando o logaritmo natural, obtemos o **Logito (*Log-Odds*)**, que é perfeitamente linear em relação aos parâmetros:

$$\ln\left( \frac{p}{1 - p} \right) = \beta_0 + \beta_1 X_1 + \dots + \beta_k X_k$$

No R, ajustamos esse modelo através da função `glm()` (*Generalized Linear Models*) especificando `family = binomial(link = "logit")`.

---

## 3. Avaliação Rigorosa de Classificadores

### 3.1 A Matriz de Confusão
Ao cruzar a realidade com a predição gerada com base em um limiar $\tau$ (padrão $0.5$):

| | Predito Negativo ($\hat{Y} = 0$) | Predito Positivo ($\hat{Y} = 1$) |
| :--- | :--- | :--- |
| **Real Negativo ($Y = 0$)** | **VN** (*Verdadeiro Negativo*) | **FP** (*Falso Positivo* - Erro Tipo I) |
| **Real Positivo ($Y = 1$)** | **FN** (*Falso Negativo* - Erro Tipo II) | **VP** (*Verdadeiro Positivo*) |

### 3.2 Indicadores Numéricos de Qualidade
* **Acurácia Geral**: Proporção total de acertos sobre todas as previsões.
  $$\text{Acurácia} = \frac{VP + VN}{VP + VN + FP + FN}$$
* **Sensitividade (*Recall* / TPR - *True Positive Rate*)**: Capacidade do modelo de detectar a classe positiva quando ela de fato ocorre.
  $$\text{Sensitividade} = \frac{VP}{VP + FN}$$
* **Especificidade (TNR - *True Negative Rate*)**: Capacidade de identificar corretamente os casos negativos.
  $$\text{Especificidade} = \frac{VN}{VN + FP}$$
* **Acurácia Balanceada**: Média aritmética entre Sensitividade e Especificidade (fundamental em dados desbalanceados).
  $$\text{Acurácia Balanceada} = \frac{\text{Sensitividade} + \text{Especificidade}}{2}$$
* **Precisão**: Dentre todas as observações classificadas como positivas, quantas realmente eram positivas.
  $$\text{Precisão} = \frac{VP}{VP + FP}$$
* **Função de Perda Logarítmica (*Log-Loss* / Entropia Cruzada Binária)**: Penaliza predições probabilísticas confiantes, mas erradas.
  $$\text{Log-Loss} = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log_2(p_i + \epsilon) + (1 - y_i) \log_2(1 - p_i + \epsilon) \right]$$

### 3.3 A Curva ROC e a Área sob a Curva (AUC)
A **Curva ROC (*Receiver Operating Characteristic*)** mapeia a **Sensitividade (TPR)** no eixo $Y$ contra a **Taxa de Falso Positivo ($FPR = 1 - \text{Especificidade}$)** no eixo $X$ para **todos os limiares de decisão possíveis entre 0 e 1**.
* A **AUC (*Area Under Curve*)** resume a performance global em um escalar entre $0.5$ (modelo aleatório, sem poder discriminatório) e $1.0$ (classificador perfeito).

---

## 4. Passo a Passo Prático: Classificação de Qualidade da Água

Aplicaremos a metodologia didática:
* 🎯 **O que fazer**
* 💻 **Como fazer**
* 💡 **Por que fazer**

### Passo 1: Ingestão de Dados, Tratamento de NAs e Análise de Desbalanceamento

🎯 **O que fazer**:
Carregar a base de potabilidade da água (`datasets/water_potability.csv`), inspecionar e imputar os valores ausentes de variáveis físico-químicas essenciais (como pH e Sulfato) e avaliar a proporção de amostras potáveis vs. não potáveis.

💻 **Como fazer**:

In [ ]:
# Limpeza do ambiente
rm(list = ls())

# Configuração gráfica
if (requireNamespace("repr", quietly = TRUE)) {
  options(repr.plot.width = 11, repr.plot.height = 5)
}

caminho_agua <- if (file.exists("../datasets/water_potability.csv")) {
  "../datasets/water_potability.csv"
} else if (file.exists("datasets/water_potability.csv")) {
  "datasets/water_potability.csv"
} else {
  stop("Arquivo water_potability.csv não encontrado!")
}

dados_agua <- read.csv(caminho_agua, stringsAsFactors = FALSE)
cat("Amostras de água carregadas:", nrow(dados_agua), "| Parâmetros físico-químicos:", ncol(dados_agua), "\n")

# 1. Tratamento de valores ausentes (imputação pela mediana de cada coluna)
for (col in names(dados_agua)) {
  if (is.numeric(dados_agua[[col]]) && any(is.na(dados_agua[[col]]))) {
    mediana_col <- median(dados_agua[[col]], na.rm = TRUE)
    dados_agua[[col]][is.na(dados_agua[[col]])] <- mediana_col
  }
}

# 2. Diagnóstico de distribuição da classe alvo (Potability)
tabela_classes <- table(dados_agua$Potability)
prop_classes <- prop.table(tabela_classes)

cat("\nDistribuição da variável dependente 'Potability':\n")
cat("Não Potável (0):", tabela_classes["0"], "(", round(prop_classes["0"] * 100, 1), "%)\n")
cat("Potável     (1):", tabela_classes["1"], "(", round(prop_classes["1"] * 100, 1), "%)\n")

💡 **Por que fazer**:
Imputar os NAs pela mediana preserva as 3.276 amostras do dataset (evitando descartar mais de um terço da base). Verificar o balanceamento das classes é crucial: bases desbalanceadas (ex.: 61% não potável vs 39% potável) tornam a acurácia simples ilusória, exigindo o uso de acurácia balanceada e AUC.

### Passo 2: Ajuste do Modelo com `glm()` e Predição de Probabilidades

🎯 **O que fazer**:
Ajustar o modelo de regressão logística com a função `glm(formula, family = binomial, data)` e gerar as probabilidades preditas $\hat{p}_i = P(Y = 1 \mid X)$ para cada amostra através de `predict(..., type = "response")`.

💻 **Como fazer**:

In [ ]:
# 1. Ajuste do modelo de Regressão Logística
# Selecionamos os principais atributos físico-químicos: ph, Sulfato, Cloraminas, Dureza, Sólidos
modelo_agua <- glm(Potability ~ ph + Hardness + Solids + Chloramines + Sulfate + Conductivity + Organic_carbon,
                    data = dados_agua, family = binomial(link = "logit"))

print(summary(modelo_agua))

# 2. Obtenção das probabilidades preditas no intervalo (0, 1)
# O argumento type = "response" garante que o retorno seja p(X) e não o log-odds z
probabilidades_agua <- predict(modelo_agua, type = "response")

cat("\nResumo das probabilidades de potabilidade calculadas:\n")
print(summary(probabilidades_agua))

💡 **Por que fazer**:
O parâmetro `type = "response"` é o detalhe técnico mais importante no R: sem ele, a função `predict()` retorna os valores na escala linear de log-odds ($z \in (-\infty, +\infty)$) em vez de probabilidades reais entre $0$ e $1$ ($0\%$ a $100\%$).

### Passo 3: Definição do Limiar (*Threshold*), Matriz de Confusão e Métricas

🎯 **O que fazer**:
Aplicar o limiar de corte $\tau = 0.5$ para converter probabilidades em classes preditas ($0$ ou $1$), construir a Matriz de Confusão com `table()` e computar programmaticamente todos os indicadores clínicos e de negócio.

💻 **Como fazer**:

In [ ]:
# 1. Aplicação do limiar padrão de 50%
limiar_corte <- 0.5
classes_preditas <- ifelse(probabilidades_agua >= limiar_corte, 1, 0)

# 2. Construção da Matriz de Confusão (Real nas linhas, Predito nas colunas)
mat_conf <- table(Real = dados_agua$Potability, Predito = classes_preditas)
print("Matriz de Confusão Absoluta:")
print(mat_conf)

# 3. Extração dos quatro quadrantes
vn <- mat_conf[1, 1] # Real 0, Predito 0
fp <- mat_conf[1, 2] # Real 0, Predito 1
fn <- mat_conf[2, 1] # Real 1, Predito 0
vp <- mat_conf[2, 2] # Real 1, Predito 1

# 4. Cálculo das Métricas
acc <- (vp + vn) / sum(mat_conf)
tpr <- vp / (vp + fn) # Sensitividade / Recall
tnr <- vn / (vn + fp) # Especificidade
acc_bal <- (tpr + tnr) / 2
prec <- vp / (vp + fp)

cat("\n--- Indicadores de Performance com Limiar 0.5 ---\n")
cat("Acurácia Geral:", round(acc * 100, 2), "%\n")
cat("Acurácia Balanceada:", round(acc_bal * 100, 2), "%\n")
cat("Sensitividade (Recall - Detectar Potável):", round(tpr * 100, 2), "%\n")
cat("Especificidade (Rejeitar Não Potável):", round(tnr * 100, 2), "%\n")
cat("Precisão:", round(prec * 100, 2), "%\n")

💡 **Por que fazer**:
Em aplicações críticas (como saúde pública ou segurança hídrica), o custo de um **Falso Negativo** (deixar passar água poluída como potável) é infinitamente mais grave que o custo de um Falso Positivo. A Matriz de Confusão desmembra onde o classificador erra, permitindo que cientistas de dados calibrem o limiar para priorizar segurança sobre conveniência.

### Passo 4: Implementação da Função de Perda (*Log-Loss*) Clássica e Balanceada

🎯 **O que fazer**:
Implementar e calcular a função de perda logarítmica (*Log-Loss* / Entropia Cruzada Binária) clássica e sua versão balanceada por classes (incorporando a lógica dos scripts utilitários do repositório).

💻 **Como fazer**:

In [ ]:
# Função de Log-Loss com estabilidade numérica (epsilon para evitar log2(0))
calc_log_loss <- function(real, pred, epsilon = 1.0e-9) {
  real <- as.numeric(real == 1)
  loss_total <- 0.0
  for (i in 1:length(real)) {
    termo_neg <- (1 - real[i]) * log2(1 - pred[i] + epsilon)
    termo_pos <- real[i] * log2(pred[i] + epsilon)
    loss_total <- loss_total + termo_neg + termo_pos
  }
  return(-loss_total / length(real))
}

# Versão balanceada que equaliza os pesos de classes desbalanceadas
calc_log_loss_balanceada <- function(real, pred) {
  real <- as.numeric(real == 1)
  loss_neg <- calc_log_loss(real[real == 0], pred[real == 0])
  loss_pos <- calc_log_loss(real[real == 1], pred[real == 1])
  return((loss_neg + loss_pos) / 2.0)
}

loss_geral <- calc_log_loss(dados_agua$Potability, probabilidades_agua)
loss_bal   <- calc_log_loss_balanceada(dados_agua$Potability, probabilidades_agua)

cat("--- Avaliação da Função de Perda (Log-Loss) ---\n")
cat("Log-Loss Geral:", round(loss_geral, 4), "bits de incerteza\n")
cat("Log-Loss Balanceada:", round(loss_bal, 4), "bits de incerteza\n")

💡 **Por que fazer**:
A acurácia avalia apenas se a classe final foi acertada, ignorando o nível de certeza. Se o modelo atribui probabilidade $0.51$ e acerta, ele ganha o mesmo ponto do que se tivesse atribuído $0.99$. A **Log-Loss** avalia a calibração probabilística real: quanto menor a perda, mais confiante e exato é o modelo.

### Passo 5: Geração da Curva ROC e Cálculo da Área sob a Curva (AUC)

🎯 **O que fazer**:
Construir a curva ROC e computar o valor de AUC utilizando a biblioteca padrão `ROCR` (ou cálculo trapezoidal direto em caso de fallback).

💻 **Como fazer**:

In [ ]:
if (requireNamespace("ROCR", quietly = TRUE)) {
  library(ROCR)
  
  # 1. Criação do objeto de predição do pacote ROCR
  pred_obj <- prediction(probabilidades_agua, dados_agua$Potability)
  
  # 2. Objeto de performance para Sensitividade (tpr) vs 1-Especificidade (fpr)
  perf_roc <- performance(pred_obj, measure = "tpr", x.measure = "fpr")
  
  # 3. Cálculo formal da Área sob a Curva (AUC)
  perf_auc <- performance(pred_obj, measure = "auc")
  auc_valor <- as.numeric(perf_auc@y.values)
  
  # 4. Plotagem da Curva ROC
  plot(perf_roc, col = "#2b5c8f", lwd = 3,
       main = paste("Curva ROC — Potabilidade da Água (AUC =", round(auc_valor, 3), ")"),
       xlab = "Taxa de Falso Positivo (1 - Especificidade)",
       ylab = "Taxa de Verdadeiro Positivo (Sensitividade / Recall)")
  # Linha diagonal de referência aleatória (AUC = 0.5)
  abline(a = 0, b = 1, col = "#e63946", lty = 2, lwd = 2)
  grid()
  legend("bottomright", legend = c(paste("Modelo Logístico (AUC =", round(auc_valor, 3), ")"), "Classificador Aleatório (AUC = 0.500)"),
         col = c("#2b5c8f", "#e63946"), lwd = c(3, 2), lty = c(1, 2))
  
  cat("Valor numérico da AUC:", round(auc_valor, 4), "\n")
} else {
  cat("Pacote ROCR não encontrado no ambiente. Instale com install.packages('ROCR') para visualização da curva.\n")
}

💡 **Por que fazer**:
A Curva ROC desvincula a avaliação do modelo de um limiar fixo (como $0.5$). Ela revela a capacidade intrínseca do classificador de separar as duas classes em qualquer cenário operacional. Um valor de AUC próximo de $0.70$ a $0.80$ comprova que o modelo aprendeu padrões químicos reais da água, enquanto a reta diagonal ($0.5$) representaria adivinhação por cara ou coroa.

## 5. Conclusão Geral da Trilha de Resumos Didáticos
Com este sexto e último caderno, encerramos a jornada completa de Ciência de Dados proposta no repositório:
1. **Módulo 1**: Fundamentos da Pirâmide DIKW, ecossistema Jupyter e o algoritmo MapReduce em Big Data.
2. **Módulo 2**: Sintaxe funcional, vetores, matrizes, fatores e data frames em R.
3. **Módulo 3.1 & 3.2**: Engenharia de dados prática (ETL, higienização de sensores, detecção de outliers via IQR de Tukey e Regex massivo).
4. **Módulo 4**: Modelagem linear simples, Mínimos Quadrados, interpretação de `summary()` e métricas MAE/MSE/RMSE.
5. **Módulo 5**: Regressão linear múltipla, diagnóstico de multicolinearidade (VIF) e padronização rigorosa de dados.
6. **Módulo 6**: Classificação supervisionada com regressão logística, matriz de confusão, acurácia balanceada, log-loss e Curva ROC / AUC.

Todos os cadernos da pasta `resumos/` foram projetados para serem **reprodutíveis, didáticos e autocontidos**, servindo como material de referência permanente para estudo e desenvolvimento profissional.